# 17. 비중 증가 노선의 격자별 기여 지도

**질문:** 역 권역에서 상대적 비중이 늘어난 광역버스 노선은, 어느 분석 격자의 해당 노선 승차기록에서 그 증가가 발생했는가?

대상: GTX-A 공급구조 유지 250m 격자 126개. 2024-10-17과 2025-11-06의 합성 교통카드자료로 구축된 **같은 126개 격자**의 셀×노선 종일 승차량을 사용한다. 2025년 10월 자료는 사용하지 않는다.

- 지도는 대곡 1000, 운정중앙 9030, 킨텍스 M7731 세 사례를 한 권역씩 보여준다.
- **노선 비중 변화 기여(%p)** = (그 격자의 해당 노선 2025년 승차 / 권역 전체 2025년 승차 − 그 격자의 해당 노선 2024년 승차 / 권역 전체 2024년 승차) × 100. 같은 노선의 모든 격자를 더하면 그 노선의 권역 내 비중 변화가 정확히 나온다.
- **격자 전체 비중 변화(%p)**는 그 격자의 모든 분석 노선 승차량을 합쳐 계산한다. 노선 기여와 격자 전체 비중은 다를 수 있다.
- 색은 노선 비중에 대한 기여 방향. 노란 테두리는 양의 기여 상위 3개를 찾기 위한 표시다. 변화의 원인을 증명하는 색이 아니다.
- 노선의 비중 증가가 곧 노선의 승차인원 증가라는 뜻은 아니다.
- 배경지도는 인터넷에서 불러온다. 인터넷이 차단되면 셀 폴리곤과 아래 수치표를 확인한다. 지도/결과 CSV/HTML 파일은 별도 저장하지 않는다.

In [1]:
# 프로젝트 루트(UROP) 자동 탐색 — 절대경로 없이 어느 컴퓨터에서도 동작
import sys
from pathlib import Path
UROP_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.py').is_file() and (p / 'code').is_dir())
for _d in [UROP_ROOT, *sorted(d for d in (UROP_ROOT / 'code').rglob('*') if d.is_dir() and d.name != '__pycache__')]:
    if str(_d) not in sys.path:
        sys.path.append(str(_d))

from pathlib import Path
import sys, numpy as np, pandas as pd, geopandas as gpd, folium
from IPython.display import display
candidates=[*sorted(d for d in (UROP_ROOT / 'code').rglob('*') if d.is_dir()), Path.cwd(),Path.cwd()/'analysis',Path.cwd()/'gtx_a_bus_analysis'/'analysis',Path.cwd()/'github_2026-UROP'/'gtx_a_bus_analysis'/'analysis']
ANALYSIS=next((x for x in candidates if (x/'ridge_count_validation.py').exists()),None)
if ANALYSIS is None:raise FileNotFoundError('프로젝트 또는 analysis 폴더에서 실행하세요.')
if str(ANALYSIS) not in sys.path:sys.path.insert(0,str(ANALYSIS))
import ridge_count_validation as source
DATA=UROP_ROOT/'data'/'processed'/'gtx_a'
route_path=next(DATA.glob('20_*.csv'))
a=source.load_data().copy()
a['셀 ID']=a['셀 ID'].astype(int)
a['격자비중변화(%p)']=100*(a.share_post-a.share_pre)
route=pd.read_csv(route_path,encoding='utf-8-sig')
route['셀 ID']=pd.to_numeric(route['셀 ID'],errors='raise').astype(int)
route['노선']=route['노선'].astype(str)
route=route.merge(a[['셀 ID','nearest_station','board_2024_all','board_2025_all','격자비중변화(%p)','walk_min']],on='셀 ID',how='inner',validate='many_to_one')
assert len(a)==126 and a['셀 ID'].nunique()==126
assert route['셀 ID'].nunique()==126 and len(route)==201
assert route[['y2024','y2025']].sum().to_numpy().tolist()==a[['board_2024_all','board_2025_all']].sum().astype(int).to_list()
assert route.groupby('셀 ID')[['y2024','y2025']].sum().rename(columns={'y2024':'board_2024_all','y2025':'board_2025_all'}).eq(a.set_index('셀 ID')[['board_2024_all','board_2025_all']].astype(int)).all().all()
totals=a.groupby('nearest_station')[['board_2024_all','board_2025_all']].sum()
route['노선비중기여(%p)']=100*(route.y2025/route.nearest_station.map(totals.board_2025_all)-route.y2024/route.nearest_station.map(totals.board_2024_all))
route['노선승차변화(명)']=route.y2025-route.y2024
summary=route.groupby(['nearest_station','노선'],as_index=False).agg(승차2024=('y2024','sum'),승차2025=('y2025','sum'),비중변화pp=('노선비중기여(%p)','sum'),승차기록격자=('셀 ID','nunique'))
summary['승차변화']=summary.승차2025-summary.승차2024
summary['권역승차2024']=summary.nearest_station.map(totals.board_2024_all)
summary['권역승차2025']=summary.nearest_station.map(totals.board_2025_all)
summary['비중2024']=summary.승차2024/summary.권역승차2024*100
summary['비중2025']=summary.승차2025/summary.권역승차2025*100
assert np.allclose(summary.비중2025-summary.비중2024,summary.비중변화pp)
print('확인: 셀×노선 합계와 126개 격자 전체 승차량이 연도별로 정확히 일치')
display(summary[summary.비중변화pp>0].sort_values(['nearest_station','비중변화pp'],ascending=[True,False])[['nearest_station','노선','승차기록격자','승차2024','승차2025','승차변화','비중2024','비중2025','비중변화pp']].round(3))

확인: 셀×노선 합계와 126개 격자 전체 승차량이 연도별로 정확히 일치


,nearest_station,노선,승차기록격자,승차2024,승차2025,승차변화,비중2024,비중2025,비중변화pp
0,대곡,1000,2,626,312,-314,23.875,30.709,6.834
2,대곡,1200,7,522,207,-315,19.908,20.374,0.466
16,운정중앙,9030,11,503,257,-246,7.749,13.448,5.699
12,운정중앙,2200,13,1083,367,-716,16.685,19.205,2.520
25,운정중앙,M7731,1,69,43,-26,1.063,2.250,1.187
18,운정중앙,9709,8,103,49,-54,1.587,2.564,0.977
11,운정중앙,1500,5,61,23,-38,0.940,1.204,0.264
23,운정중앙,M7111,2,224,70,-154,3.451,3.663,0.212
20,운정중앙,9710-1,1,4,4,0,0.062,0.209,0.148
34,킨텍스,M7731,6,3102,1217,-1885,35.326,45.786,10.460


## 지도 읽는 법

빨간 격자: 해당 노선의 **권역 내 비중 증가에 양으로 기여**. 파란 격자: 음으로 기여. 회색: 해당 권역의 다른 분석 격자. 노란 외곽선: 기여 상위 3개. 격자를 클릭하면 노선 승차량·기여값·격자 전체 비중 변화·GTX역 보행시간을 볼 수 있다.

양의 기여값을 가진 격자도 해당 노선 승차량은 줄었을 수 있다. 권역 전체 승차량이 더 크게 감소하면 노선 비중 기여가 양수가 된다. 지도는 공간적 위치와 기여를 보여주며, 접근성이 그 차이를 유발했다는 결론은 내리지 않는다.

In [2]:
geo=source.attach_geometry(a).to_crs('EPSG:4326')
assert len(geo)==126 and geo['셀 ID'].is_unique and geo.geometry.notna().all()
STATIONS={'운정중앙':(37.716670,126.728330),'킨텍스':(37.665000,126.748060),'대곡':(37.631626,126.811024)}
FOCUS=[('대곡','1000'),('운정중앙','9030'),('킨텍스','M7731')]

def draw_map(station,number):
    part=route[(route.nearest_station==station)&(route['노선']==number)].copy()
    if part.empty:raise ValueError(f'{station} {number}: 셀×노선 행 없음')
    top=set(part.nlargest(min(3,(part['노선비중기여(%p)']>0).sum()),'노선비중기여(%p)')['셀 ID'])
    layer=geo[geo.nearest_station==station]
    display_map=folium.Map(location=STATIONS[station],zoom_start=13,tiles='CartoDB positron',control_scale=True)
    # 권역 안의 분석 격자를 먼저 회색으로 표시한다.
    for _,cell in layer.iterrows():
        coords=[(lat,lon) for lon,lat in cell.geometry.exterior.coords]
        folium.Polygon(coords,color='#A0A0A0',weight=1,fill=True,fill_color='#E6E6E6',fill_opacity=.25,tooltip=f"분석 셀 {int(cell['셀 ID'])}").add_to(display_map)
    merged=gpd.GeoDataFrame(part.merge(layer[['셀 ID','geometry']],on='셀 ID',validate='one_to_one'),geometry='geometry',crs=layer.crs)
    for _,cell in merged.iterrows():
        delta=cell['노선비중기여(%p)']; positive=delta>0
        coords=[(lat,lon) for lon,lat in cell.geometry.exterior.coords]
        info=(f"셀 {int(cell['셀 ID'])}<br>노선 {number}: {int(cell.y2024)}→{int(cell.y2025)}명"
              f"<br>노선 비중 기여: {delta:+.3f}%p"
              f"<br>격자 전체 비중 변화: {cell['격자비중변화(%p)']:+.3f}%p"
              f"<br>GTX 보행시간: {cell.walk_min:.1f}분")
        folium.Polygon(coords,color='#E6B800' if cell['셀 ID'] in top else '#222222',
                       weight=5 if cell['셀 ID'] in top else 1.5,fill=True,
                       fill_color='#D73027' if positive else '#2B8CBE',fill_opacity=.70,
                       tooltip=folium.Tooltip(info),popup=folium.Popup(info,max_width=340)).add_to(display_map)
    folium.CircleMarker(STATIONS[station],radius=7,color='#008B4A',fill=True,
                        fill_color='#008B4A',tooltip='GTX-A역').add_to(display_map)
    y0,x0,y1,x1=merged.total_bounds[[1,0,3,2]]
    pad=.006
    display_map.fit_bounds([[y0-pad,x0-pad],[y1+pad,x1+pad]])
    title=f"{station} · {number}번 | 빨강: 양의 기여, 파랑: 음의 기여, 노랑: 상위 3격자"
    print(title)
    display(display_map)
    table=part.sort_values('노선비중기여(%p)',ascending=False)[['셀 ID','y2024','y2025','노선승차변화(명)','노선비중기여(%p)','board_2024_all','board_2025_all','격자비중변화(%p)','walk_min']].copy()
    table=table.rename(columns={'y2024':'노선2024승차','y2025':'노선2025승차','board_2024_all':'격자전체2024승차','board_2025_all':'격자전체2025승차','walk_min':'GTX보행분'})
    display(table.round(3).reset_index(drop=True))
    print('이 노선의 전체 기여 합:',round(part['노선비중기여(%p)'].sum(),3),'%p | 격자 수:',len(part),'| 노선 승차 변화:',int(part['노선승차변화(명)'].sum()),'명')
    return display_map,table

MAPS={};TABLES={}
for station,number in FOCUS:
    MAPS[(station,number)],TABLES[(station,number)]=draw_map(station,number)

대곡 · 1000번 | 빨강: 양의 기여, 파랑: 음의 기여, 노랑: 상위 3격자
이 노선의 전체 기여 합: 6.834 %p | 격자 수: 2 | 노선 승차 변화: -314 명
운정중앙 · 9030번 | 빨강: 양의 기여, 파랑: 음의 기여, 노랑: 상위 3격자
이 노선의 전체 기여 합: 5.699 %p | 격자 수: 11 | 노선 승차 변화: -246 명
킨텍스 · M7731번 | 빨강: 양의 기여, 파랑: 음의 기여, 노랑: 상위 3격자
이 노선의 전체 기여 합: 10.46 %p | 격자 수: 6 | 노선 승차 변화: -1885 명


,셀 ID,노선2024승차,노선2025승차,노선승차변화(명),노선비중기여(%p),격자전체2024승차,격자전체2025승차,격자비중변화(%p),GTX보행분
0,12461,380,193,-187,4.503,595.0,283.0,5.162,40.322
1,12464,246,119,-127,2.330,277.0,129.0,2.132,44.312


,셀 ID,노선2024승차,노선2025승차,노선승차변화(명),노선비중기여(%p),격자전체2024승차,격자전체2025승차,격자비중변화(%p),GTX보행분
0,10778,94,98,4,3.680,1123.0,381.0,2.636,31.372
1,7697,38,23,-15,0.618,231.0,53.0,-0.785,18.593
2,13084,91,36,-55,0.482,320.0,88.0,-0.325,15.390
3,14103,64,28,-36,0.479,289.0,62.0,-1.208,19.333
4,11811,67,28,-39,0.433,247.0,71.0,-0.090,30.455
5,7480,37,15,-22,0.215,93.0,39.0,0.608,36.747
6,5964,17,7,-10,0.104,17.0,7.0,0.104,39.873
7,12556,30,10,-20,0.061,65.0,21.0,0.098,28.575
8,9146,10,4,-6,0.055,10.0,4.0,0.055,49.578
9,3056,16,1,-15,-0.194,16.0,1.0,-0.194,39.670


,셀 ID,노선2024승차,노선2025승차,노선승차변화(명),노선비중기여(%p),격자전체2024승차,격자전체2025승차,격자비중변화(%p),GTX보행분
0,4934,867,412,-455,5.627,1990.0,608.0,0.212,20.180
1,12765,874,346,-528,3.064,882.0,373.0,3.989,3.553
2,9342,236,150,-86,2.956,236.0,150.0,2.956,26.005
3,14062,18,18,0,0.472,21.0,23.0,0.626,38.118
4,11506,71,25,-46,0.132,71.0,25.0,0.132,45.487
5,8868,1036,266,-770,-1.791,1036.0,266.0,-1.791,7.923


## 사례 해석과 다음 확인

- **대곡 1000번:** 126개 분석 격자 중 대곡 권역에서 1000번 승차기록이 있는 2개 격자 모두 비중 증가에 기여하는지 확인한다. 두 격자의 비중 변화 합이 노선 전체 +6.834%p와 일치한다.
- **운정중앙 9030번:** 셀 10778 한 곳이 크게 기여하지만, 같은 노선 승차기록이 있는 일부 격자는 전체 격자 비중이 오히려 감소할 수 있다.
- **킨텍스 M7731:** 양의 기여 격자와 음의 기여 격자가 함께 있어 노선 승차기록이 있다는 사실만으로 격자 결과를 판단할 수 없다.

다음에 해석할 때는 노선의 목적지/경유지, 승차정류장, GTX역까지의 실제 접근 및 주변 시설을 **확인한 사실만** 지도와 연결한다. 2024·2025년이 각각 하루의 합성자료이며, 노선별 이용자 전환을 직접 관찰한 것은 아니다. GTX-C의 노선 비중 증가 여부는 아직 알 수 없다.